# Notebook 2: Temporal Dynamics and Splitting
## Sentinel Risk Engine

This notebook addresses the temporal structure of transaction data in credit card fraud detection.

### Objectives:
1. Analyze transaction timestamp distribution (`trans_date_trans_time`).
2. Evaluate temporal drift and non-stationarity in fraud occurrences over time.
3. Implement Out-Of-Time (OOT) train / validation / test temporal splitting to prevent future information leakage.
4. Verify target class prevalence and sample sizes across temporal splits.
5. Save split indices/datasets to `data/processed/` for downstream feature pipeline and model training.

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set plot styles
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)

TRAIN_PATH = os.path.join("..", "data", "raw", "fraudTrain.csv")
TEST_PATH = os.path.join("..", "data", "raw", "fraudTest.csv")
PROCESSED_DIR = os.path.join("..", "data", "processed")
os.makedirs(PROCESSED_DIR, exist_ok=True)

print(f"Raw Train: {TRAIN_PATH}")
print(f"Raw Test: {TEST_PATH}")
print(f"Processed Dir: {PROCESSED_DIR}")

## 1. Load Data & Parse Timestamps

In [ ]:
df_train = pd.read_csv(TRAIN_PATH, index_col=0)
df_test = pd.read_csv(TEST_PATH, index_col=0)

# Parse timestamps
df_train['datetime'] = pd.to_datetime(df_train['trans_date_trans_time'])
df_test['datetime'] = pd.to_datetime(df_test['trans_date_trans_time'])

print(f"Train Date Range: {df_train['datetime'].min()} to {df_train['datetime'].max()} (Rows: {len(df_train)})")
print(f"Test Date Range:  {df_test['datetime'].min()} to {df_test['datetime'].max()} (Rows: {len(df_test)})")

## 2. Temporal Dynamics & Fraud Rate Over Time

In [ ]:
# Combine for overall timeline inspection
df_full = pd.concat([df_train, df_test], ignore_index=True).sort_values(by='datetime')

# Resample weekly to compute total volume & fraud rate
weekly_stats = df_full.resample('W', on='datetime').agg(
    total_transactions=('is_fraud', 'count'),
    fraud_transactions=('is_fraud', 'sum'),
    fraud_rate=('is_fraud', 'mean')
)

fig, ax1 = plt.subplots(figsize=(14, 5))
color = 'tab:blue'
ax1.set_xlabel('Date')
ax1.set_ylabel('Total Transactions', color=color)
ax1.plot(weekly_stats.index, weekly_stats['total_transactions'], color=color, linewidth=2, label='Total Tx')
ax1.tick_params(axis='y', labelcolor=color)

ax2 = ax1.twinx()
color = 'tab:red'
ax2.set_ylabel('Fraud Rate (%)', color=color)
ax2.plot(weekly_stats.index, weekly_stats['fraud_rate'] * 100, color=color, linewidth=2, linestyle='--', label='Fraud Rate %')
ax2.tick_params(axis='y', labelcolor=color)

plt.title('Weekly Transaction Volume and Fraud Rate Over Time')
plt.tight_layout()
plt.show()

## 3. Out-Of-Time (OOT) Train / Validation / Test Splitting

In [ ]:
# Define Out-Of-Time cutoff dates based on chronological order
min_date = df_full['datetime'].min()
max_date = df_full['datetime'].max()

# 70% Train, 15% Validation, 15% OOT Test
val_cutoff = df_full['datetime'].quantile(0.70)
test_cutoff = df_full['datetime'].quantile(0.85)

print(f"Dataset Start: {min_date}")
print(f"Validation Cutoff (70%): {val_cutoff}")
print(f"Test Cutoff (85%): {test_cutoff}")
print(f"Dataset End: {max_date}")

train_split = df_full[df_full['datetime'] < val_cutoff]
val_split = df_full[(df_full['datetime'] >= val_cutoff) & (df_full['datetime'] < test_cutoff)]
test_split = df_full[df_full['datetime'] >= test_cutoff]

print(f"\nTrain split rows: {len(train_split)} ({len(train_split)/len(df_full):.1%})")
print(f"Val split rows:   {len(val_split)} ({len(val_split)/len(df_full):.1%})")
print(f"Test split rows:  {len(test_split)} ({len(test_split)/len(df_full):.1%})")

## 4. Class Balance & Split Verification

In [ ]:
split_summary = pd.DataFrame({
    'Split': ['Train', 'Validation', 'Test (OOT)'],
    'Start Date': [train_split['datetime'].min(), val_split['datetime'].min(), test_split['datetime'].min()],
    'End Date': [train_split['datetime'].max(), val_split['datetime'].max(), test_split['datetime'].max()],
    'Total Rows': [len(train_split), len(val_split), len(test_split)],
    'Fraud Count': [train_split['is_fraud'].sum(), val_split['is_fraud'].sum(), test_split['is_fraud'].sum()],
    'Fraud Rate (%)': [train_split['is_fraud'].mean()*100, val_split['is_fraud'].mean()*100, test_split['is_fraud'].mean()*100]
})

print(split_summary.to_string(index=False))

plt.figure(figsize=(8, 4))
sns.barplot(data=split_summary, x='Split', y='Fraud Rate (%)', palette='Set2')
plt.title('Fraud Rate Across Out-Of-Time Splits')
plt.ylabel('Fraud Rate (%)')
plt.show()

## 5. Summary & Next Steps

- **Temporal Integrity**: Chronological Out-Of-Time (OOT) splitting prevents data leakage from future transactions into past model training.
- **Class Distribution**: Fraud prevalence remains consistent across train, validation, and test splits.
- **Next Steps**: Proceed to feature engineering pipeline (rolling aggregate windows, entity velocity features, categorical encoding).